# Waze IA - correccion con Node y Tree


La solucion conserva el mapa real y el costo compuesto, pero representa cada estado como un objeto `OSMRouteNode(Node)` y ejecuta A* mediante `OSMRouteTree(Tree)`.

## 1. Importaciones y mapa

La red se descarga como un grafo dirigido para vehículos. NetworkX solo se usa para representar y consultar el grafo: la búsqueda de rutas se implementa más adelante con A*.


In [ ]:
from pathlib import Path
from urllib.parse import urlencode
import heapq
import math
import time

import geopandas as gpd
import ipywidgets as widgets
import matplotlib.pyplot as plt
import osmnx as ox
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, clear_output, display
from shapely.geometry import LineString

pd.set_option("display.max_columns", None)


In [ ]:
G = ox.graph_from_place('La Estrella, Antioquia, Colombia', network_type='drive', simplify=False)
ox.plot_graph(G, figsize=(25, 25))

## 2. Información del mapa y tiempos de viaje

Se convierten los nodos y aristas a tablas geográficas y se calculan velocidades y tiempos de viaje, igual que en el ejemplo de clase.


In [ ]:
hwy_speeds = {
    'motorway': 80,
    'trunk': 60,
    'primary': 50,
    'secondary': 40,
    'tertiary': 35,
    'residential': 30,
    'unclassified': 30,
    'service': 20
}

hwy_speeds

Calcular velocidades, tiempos de viaje y orientaciones

In [ ]:
G = ox.add_edge_speeds(G, hwy_speeds=hwy_speeds)
G = ox.add_edge_travel_times(G)
G = ox.add_edge_bearings(G)

In [ ]:
gdf_nodes, gdf_edges = ox.graph_to_gdfs(G)
gdf_edges.head()

In [ ]:
gdf_edges.explode('highway').groupby('highway')[['length', 'speed_kph', 'travel_time']].mean().round(1)

## 3. Lugares estandarizados y seleccion general

La ruta no queda fijada a un par de lugares. El selector al final del cuaderno usa coordenadas estandarizadas, por lo que A*, Google Maps y Waze pueden recibir el mismo punto geografico.


In [ ]:
# Menu de lugares estandarizados. Las coordenadas son la fuente de verdad.
LUGARES_VALIDOS = {
    "Comfama": {
        "nombre": "Parque Comfama La Estrella",
        "lat": 6.1543088,
        "lon": -75.6531261,
    },
    "Hospital": {
        "nombre": "Hospital La Estrella",
        "lat": 6.1554390,
        "lon": -75.6440655,
    },
    "Cocorollo": {
        "nombre": "Cocorollo La Estrella",
        "lat": 6.1171245,
        "lon": -75.6303483,
    },
    "Pitriza":{
        "nombre": "Pitriza La Estrella",
        "lat": 6.15652,
        "lon": -75.63587,
    },
    "Plaza 77":{
        "nombre": "Plaza 77 La Estrella",
        "lat": 6.15682,
        "lon": -75.62691,
    },
    "Parque Principal":{
        "nombre": "Parque Principal La Estrella",
        "lat": 6.1576801,
        "lon": -75.6434012,
    },
    "Tablaza": {
        "nombre": "Centro de La Tablaza",
        "lat": 6.1181940,
        "lon": -75.6351392,
    },
}

MAX_SNAP_DISTANCE_M = 250
MAP_BOUNDS = gdf_nodes.total_bounds  # min_lon, min_lat, max_lon, max_lat
pd.DataFrame.from_dict(LUGARES_VALIDOS, orient="index")


La seleccion de A y B se realiza mediante el menu interactivo de la seccion 8.

## 4. Clases Node y OSMRouteNode

`Node` conserva los atributos y metodos que ya teníamos. `OSMRouteNode` hereda de ella y especializa estado, operadores, costo y heuristica para OpenStreetMap.

In [ ]:
class Node:

    def __init__(
        self, state, value, operators=None, operator=None,
        parent=None, objective=None,
    ):
        self.state = state
        self.value = value
        self.children = []
        self.parent = parent
        self.operator = operator
        self.operators = [] if operators is None else operators
        self.objective = objective
        self.level = 0
        self.v = 0

    def add_child(self, value, state, operator):
        node = type(self)(
            value=value,
            state=state,
            operator=operator,
            parent=self,
            operators=self.operators,
            objective=self.objective,
        )
        node.level = self.level + 1
        self.children.append(node)
        return node

    def add_node_child(self, node):
        node.level = node.parent.level + 1
        self.children.append(node)
        return node

    def getchildrens(self):
        children = []
        for index, _ in enumerate(self.operators):
            state = self.getState(index)
            if state is not None and not self.repeatStatePath(state):
                children.append((index, state))
        return children

    def getState(self, index):
        raise NotImplementedError("La clase hija debe implementar getState()")

    def __eq__(self, other):
        if not isinstance(other, Node):
            return NotImplemented
        return self.state == other.state

    def __lt__(self, other):
        return self.f() < other.f()

    def repeatStatePath(self, state):
        node = self
        while node is not None and node.state != state:
            node = node.parent
        return node is not None

    def pathObjective(self):
        node = self
        result = []
        while node is not None:
            result.append(node)
            node = node.parent
        return result

    def heuristic(self):
        return 0.0

    def cost(self):
        return 1.0

    def f(self):
        return self.cost() + self.heuristic()

    def isObjective(self, endState):
        return self.state == endState


class OSMRouteNode(Node):
    """Especializacion de Node para rutas dirigidas de OpenStreetMap.

    state = (nodo_anterior, nodo_actual, clave_arista_de_llegada)
    El estado ampliado permite cobrar correctamente el costo de un giro.
    """

    graph = None
    config = None
    max_speed_kph = None

    # Penalizaciones y factores de tiempo definidos en la documentacion.
    DEFAULT_CONFIG = {
        "highway_speeds_kph": {
            "motorway": 80.0, "trunk": 60.0, "primary": 50.0,
            "secondary": 40.0, "tertiary": 35.0, "residential": 30.0,
            "unclassified": 30.0, "service": 20.0, "trunk_link": 40.0,
            "primary_link": 40.0, "secondary_link": 30.0,
            "tertiary_link": 25.0,
        },
        "default_speed_kph": 30.0,
        "highway_time_factors": {
            "motorway": 1.00, "trunk": 1.00, "primary": 1.00,
            "secondary": 1.02, "tertiary": 1.05, "residential": 1.10,
            "unclassified": 1.08, "service": 1.15, "trunk_link": 1.02,
            "primary_link": 1.02, "secondary_link": 1.04,
            "tertiary_link": 1.06,
        },
        "default_highway_time_factor": 1.05,
        "intersection_delay_s_by_street_count": {3: 2.0, 4: 4.0, 5: 6.0},
        "straight_threshold_deg": 30.0,
        "u_turn_threshold_deg": 150.0,
        "turn_delay_s": {"recto": 0.0, "giro": 3.0, "retorno": 12.0},
    }

    @classmethod
    def configure(cls, graph, config=None):
        """Configura los datos compartidos por todos los nodos de la busqueda.

        Combina la configuracion predeterminada con el perfil seleccionado,
        valida sus penalizaciones, guarda el grafo y obtiene la velocidad maxima
        que se utilizara para calcular la heuristica temporal.
        """
        merged = dict(cls.DEFAULT_CONFIG)
        if config:
            merged.update(config)
        cls._validate_config(merged)
        cls.graph = graph
        cls.config = merged
        speeds = []
        for _, _, data in graph.edges(data=True):
            speeds.extend(cls._as_numbers(data.get("speed_kph")))
        if not speeds:
            raise ValueError("El grafo no contiene valores validos de speed_kph.")
        cls.max_speed_kph = max(speeds)

    def __init__(self, accumulated_cost=0.0, step_edge=None, step_detail=None, **kwargs):
        """Inicializa un estado de la busqueda aplicado al mapa vial.

        Conserva los atributos heredados de Node, el costo acumulado desde la
        raiz, la arista usada para llegar y su desglose de costo. Finalmente,
        carga como operadores todas las aristas que salen del nodo actual.
        """
        super().__init__(**kwargs)
        if self.graph is None or self.config is None:
            raise ValueError("Ejecute OSMRouteNode.configure(G, config) antes de crear la raiz.")
        self.accumulated_cost = float(accumulated_cost)
        self.step_edge = step_edge
        self.step_detail = step_detail
        self.operators = self.findAdjacents()

    @staticmethod
    def _as_numbers(value):
        """Convierte un valor OSM simple o multiple en numeros finitos.

        Los elementos vacios, no numericos o infinitos se descartan. El metodo
        siempre devuelve una lista, aunque reciba un unico valor.
        """
        values = value if isinstance(value, (list, tuple, set)) else [value]
        result = []
        for item in values:
            try:
                number = float(item)
            except (TypeError, ValueError):
                continue
            if math.isfinite(number):
                result.append(number)
        return result

    @staticmethod
    def _as_osm_values(value):
        """Normaliza un atributo OSM para poder recorrerlo como una lista."""
        return list(value) if isinstance(value, (list, tuple, set)) else [value]

    @staticmethod
    def _validate_config(config):
        """Comprueba que las demoras y factores del modelo sean validos.

        Las demoras deben ser finitas y no negativas. Los factores de tipo de
        via deben ser finitos y mayores o iguales a uno para no reducir el
        travel_time original.
        """
        delays = list(config["intersection_delay_s_by_street_count"].values())
        delays += list(config["turn_delay_s"].values())
        factors = list(config["highway_time_factors"].values())
        factors.append(config["default_highway_time_factor"])
        if any(not math.isfinite(float(x)) or float(x) < 0 for x in delays):
            raise ValueError("Todas las demoras deben ser finitas y no negativas.")
        if any(not math.isfinite(float(x)) or float(x) < 1 for x in factors):
            raise ValueError("Los factores de via deben ser finitos y mayores o iguales a 1.")

    def findAdjacents(self):
        """Obtiene las aristas dirigidas que pueden recorrerse desde el nodo actual.

        Devuelve tuplas (vecino, clave, datos). La clave permite conservar y
        distinguir aristas paralelas entre los mismos nodos.
        """
        current = self.state[1]
        return [
            (neighbor, key, data)
            for _, neighbor, key, data
            in self.graph.out_edges(current, keys=True, data=True)
        ]

    def getState(self, index):
        """Construye el estado producido por el operador indicado.

        El nuevo estado es (nodo_actual, nodo_vecino, clave_arista), de modo que
        el siguiente nodo pueda recordar por cual arista llego y calcular giros.
        """
        current = self.state[1]
        neighbor, key, _ = self.operators[index]
        return (current, neighbor, key)

    def isObjective(self, endState):
        """Indica si el nodo vial actual coincide con el destino buscado."""
        return self.state[1] == endState

    def cost(self):
        """Devuelve g(n), el costo compuesto acumulado desde la raiz."""
        return self.accumulated_cost

    def heuristic(self):
        """Calcula h(n), una estimacion del tiempo restante en segundos.

        Usa Haversine para medir la distancia sobre la superficie terrestre entre
        el nodo actual y el destino, y la divide por la velocidad maxima del
        grafo. Al usar distancia directa y velocidad maxima, no sobreestima el
        costo vial esperado y puede utilizarse como heuristica de A*.
        """
        current = self.graph.nodes[self.state[1]]
        destination = self.graph.nodes[self.objective]
        lat1, lon1 = math.radians(current["y"]), math.radians(current["x"])
        lat2, lon2 = math.radians(destination["y"]), math.radians(destination["x"])
        dlat, dlon = lat2 - lat1, lon2 - lon1
        a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
        a = min(1.0, max(0.0, a))
        distance_m = 2 * 6_371_008.8 * math.asin(math.sqrt(a))
        return distance_m * 3.6 / self.max_speed_kph

    def _highway_factor(self, edge_data):
        """Obtiene el factor de penalizacion asociado al tipo de via.

        Si una arista tiene varias clasificaciones highway, utiliza el factor mas
        alto. Para tipos desconocidos aplica el factor predeterminado.
        """
        values = self._as_osm_values(edge_data.get("highway"))
        return max(
            (float(self.config["highway_time_factors"].get(str(x), self.config["default_highway_time_factor"])) for x in values),
            default=float(self.config["default_highway_time_factor"]),
        )

    def _intersection_delay(self, node_id):
        """Calcula la demora por llegar a una interseccion.

        Selecciona la penalizacion correspondiente al street_count del nodo. No
        penaliza el destino ni nodos sin un street_count valido.
        """
        if node_id == self.objective:
            return 0.0
        try:
            street_count = int(self.graph.nodes[node_id].get("street_count", 0))
        except (TypeError, ValueError):
            return 0.0
        applicable = [int(k) for k in self.config["intersection_delay_s_by_street_count"] if street_count >= int(k)]
        return 0.0 if not applicable else float(self.config["intersection_delay_s_by_street_count"][max(applicable)])

    def _turn_delay(self, outgoing_data):
        """Clasifica el movimiento y calcula su demora.

        Compara el bearing de entrada con el de salida para reconocer un movimiento
        recto, giro o retorno. Devuelve (demora, movimiento, angulo). El inicio,
        la continuidad fuera de intersecciones y la falta de datos no se penalizan.
        """
        previous, current, incoming_key = self.state
        if previous is None:
            return 0.0, "inicio", None
        if self.graph.nodes[current].get("street_count", 0) < 3:
            return 0.0, "continuidad", None
        incoming_data = self.graph.edges[previous, current, incoming_key]
        incoming = self._as_numbers(incoming_data.get("bearing"))
        outgoing = self._as_numbers(outgoing_data.get("bearing"))
        if not incoming or not outgoing:
            return 0.0, "sin_dato", None
        angle = abs((outgoing[0] - incoming[0] + 180.0) % 360.0 - 180.0)
        if angle < self.config["straight_threshold_deg"]:
            movement = "recto"
        elif angle <= self.config["u_turn_threshold_deg"]:
            movement = "giro"
        else:
            movement = "retorno"
        return float(self.config["turn_delay_s"].get(movement, 0.0)), movement, angle

    # Recibe index, que indica cuál de los operadores (aristas salientes) se quiere utilizar
    def transition_cost(self, index):
        """Calcula el costo de recorrer la arista elegida por un operador.

        Suma travel_time, ajuste por tipo de via, demora de la interseccion de
        llegada y demora del giro. Devuelve el costo del tramo, un diccionario
        explicativo y la identidad exacta de la arista (origen, destino, clave).
        """

        # Obtiene el nodo actual
        current = self.state[1]
        # Selecciona la arista de salida
        neighbor, key, edge_data = self.operators[index]
        # Obtiene el tiempo base, es una lista porque algunos atributos de OpenStreetMap pueden contener varios valores
        times = self._as_numbers(edge_data.get("travel_time"))
        if not times:
            raise ValueError(f"La arista ({current}, {neighbor}, {key}) no tiene travel_time valido.")
        base_time = times[0]
        # Obtiene el factor del tipo de vía
        factor = self._highway_factor(edge_data)
        # Calcula los ajustes por tipo de vía, intersección y giro.
        highway_adjustment = base_time * (factor - 1.0)
        intersection_adjustment = self._intersection_delay(neighbor)
        turn_adjustment, movement, angle = self._turn_delay(edge_data)
        # Suma todos los costos para obtener el costo total del tramo y prepara un diccionario con el detalle de cada componente.
        total = base_time + highway_adjustment + intersection_adjustment + turn_adjustment
        detail = {
            "base_time_s": base_time,
            "highway_adjustment_s": highway_adjustment,
            "highway_time_factor": factor,
            "intersection_adjustment_s": intersection_adjustment,
            "turn_adjustment_s": turn_adjustment,
            "total_edge_cost_s": total,
            "movement": movement,
            "turn_angle_deg": angle,
            "street_count_arrival": self.graph.nodes[neighbor].get("street_count", 0),
            "highway": ", ".join(map(str, self._as_osm_values(edge_data.get("highway")))),
            "speed_kph": self._as_numbers(edge_data.get("speed_kph"))[0],
            "length_m": float(edge_data.get("length", 0.0)),
        }
        return total, detail, (current, neighbor, key)

    def add_child(self, value, state, operator):
        """Crea un OSMRouteNode hijo al aplicar un operador.

        Calcula el nuevo tramo, suma su costo al g(n) del padre, conserva la arista
        y su detalle, enlaza el hijo con el padre y actualiza su nivel en el arbol.
        """
        step_cost, detail, edge = self.transition_cost(operator)
        node = type(self)(
            value=value,
            state=state,
            operator=operator,
            parent=self,
            objective=self.objective,
            accumulated_cost=self.cost() + step_cost,
            step_edge=edge,
            step_detail=detail,
        )
        node.level = self.level + 1
        self.children.append(node)
        return node


Cada nodo guarda padre, hijos, operador, nivel, costo acumulado y heuristica. El estado `(anterior, actual, clave)` permite calcular el giro de llegada.

## 5. Clase Tree y ejecucion de A*

`Tree.aStar()` conserva la frontera priorizada por `f(n) = g(n) + h(n)`. Se agrega `best_g` porque el mapa vial es un grafo grande y puede alcanzar el mismo estado por caminos de distinto costo.

In [ ]:
def validate_composite_route(result, start_id, end_id):
    """Verifica la ruta producida por A* antes de mostrarla."""
    if not result["found"]:
        raise ValueError(result["message"])

    nodes = result["route_nodes"]
    edges = result["route_edges"]
    details = result["edge_details"]
    if nodes[0] != start_id or nodes[-1] != end_id:
        raise AssertionError("La ruta no coincide con los nodos seleccionados.")
    if len(edges) != len(nodes) - 1 or len(details) != len(edges):
        raise AssertionError("La estructura de la ruta es inconsistente.")
    if any(detail["total_edge_cost_s"] < 0 for detail in details):
        raise AssertionError("La ruta contiene un costo negativo.")
    if not math.isclose(
        sum(detail["total_edge_cost_s"] for detail in details),
        result["total_cost_s"], rel_tol=1e-10, abs_tol=1e-7,
    ):
        raise AssertionError("El costo total no coincide con la suma de tramos.")


def summarize_composite_result(result):
    details = pd.DataFrame(result["edge_details"])
    return pd.Series({
        "Distancia total (km)": details["length_m"].sum() / 1000,
        "Tiempo base (min)": details["base_time_s"].sum() / 60,
        "Costo estimado A* (min)": result["total_cost_s"] / 60,
        "Ajuste tipo de via (s)": details["highway_adjustment_s"].sum(),
        "Demora intersecciones (s)": details["intersection_adjustment_s"].sum(),
        "Demora giros (s)": details["turn_adjustment_s"].sum(),
        "Nodos explorados": result["explored_nodes"],
        "Tiempo de busqueda (s)": result["elapsed_time_s"],
    }).round(3)


In [ ]:
class Tree:
    """Arbol generico de busqueda basado en la clase de IA_Busquedas_Unificado."""

    def __init__(self, root, operators=None):
        self.root = root
        if operators is not None and not self.root.operators:
            self.root.operators = operators
        self.explored_nodes = set()
        self.explored_states = 0
        self.elapsed_time_s = 0.0

    def printPath(self, node):
        path = node.pathObjective()
        for current in reversed(path):
            print(current.state)
        return path

    def reinitRoot(self, endState):
        self.root.operator = None
        self.root.parent = None
        self.root.objective = endState
        self.root.children = []
        self.root.level = 0

    def aStar(self, endState):
        """A*: expande primero el Node con menor f(n) = g(n) + h(n)."""
        self.reinitRoot(endState)
        # Para estimar la duracion de la busqueda
        started_at = time.perf_counter()
        # Cola de prioridad de nodos pendientes, ordenados por f(n) y g(n)
        pending = []
        # Orden de llegada de los nodos a la cola, para romper empates en f(n) y g(n)
        order = 0
        # best_g guarda el menor costo conocido para llegar a cada estado
        best_g = {self.root.state: self.root.cost()}
        # Reiniciar las métricas
        self.explored_nodes = set()
        self.explored_states = 0
        # Se inserta la raiz en la cola de prioridad con su f(n), g(n) y orden de llegada
        heapq.heappush(pending, (self.root.f(), self.root.cost(), order, self.root))

        # Mientras la cola tenga elementos.
        while pending:
            # Se extrae el nodo con menor f(n) y g(n) de la cola de prioridad.
            _, queued_g, _, node = heapq.heappop(pending)
            # Si el g(n) del nodo extraido es mayor que el mejor g(n) conocido para su estado, se descarta.
            if queued_g > best_g.get(node.state, math.inf):
                continue
            self.explored_states += 1
            # Obtiene el identificador del nodo actual (anterior, [actual], clave_arista) y lo agrega al conjunto de nodos explorados.
            current_id = node.state[1] if isinstance(node.state, tuple) else node.state
            self.explored_nodes.add(current_id)
            # Comprueba si el nodo vial actual es el destino
            if node.isObjective(endState):
                self.elapsed_time_s = time.perf_counter() - started_at
                return node

            # Obtiene los posibles movimientos consultando los operadores de OSMRouteNode.
            for operator_index, child_state in node.getchildrens():
                # Se calcula el costo por tramo y acumulado internamente.
                new_child = node.add_child(
                    value=node.value + "-" + str(operator_index),
                    state=child_state,
                    operator=operator_index,
                )
                # Comprueba si el camino nuevo es mejor.
                if new_child.cost() < best_g.get(child_state, math.inf):
                    # Actualiza el mejor costo conocido para llegar a ese estado y agrega el hijo a la cola de prioridad.
                    best_g[child_state] = new_child.cost()
                    order += 1
                    heapq.heappush(
                        pending,
                        (new_child.f(), new_child.cost(), order, new_child),
                    )

        self.elapsed_time_s = time.perf_counter() - started_at
        return None


class OSMRouteTree(Tree):
    """Nombre especializado para hacer explicita la relacion Tree -> mapa vial."""
    pass


def result_from_goal(goal_node, tree):
    """Adapta el Node objetivo al formato usado por tablas y visualizaciones."""
    if goal_node is None:
        return {
            "found": False, "route_nodes": [], "route_edges": [],
            "edge_details": [], "total_cost_s": math.inf,
            "explored_nodes": len(tree.explored_nodes),
            "explored_states": tree.explored_states,
            "elapsed_time_s": tree.elapsed_time_s,
            "message": "No existe una ruta dirigida entre el origen y el destino.",
        }

    path = list(reversed(goal_node.pathObjective()))
    return {
        "found": True,
        "route_nodes": [node.state[1] for node in path],
        "route_edges": [node.step_edge for node in path[1:]],
        "edge_details": [node.step_detail for node in path[1:]],
        "total_cost_s": goal_node.cost(),
        "explored_nodes": len(tree.explored_nodes),
        "explored_states": tree.explored_states,
        "elapsed_time_s": tree.elapsed_time_s,
        "message": "Ruta A* encontrada mediante Node y Tree.",
    }


La clase especializada `OSMRouteTree(Tree)` hace explicita la arquitectura exigida, mientras `result_from_goal` adapta el nodo objetivo a las tablas y mapas existentes.

In [ ]:
perfiles_costo = {
    "1. Solo travel_time": {
        "highway_time_factors": {},
        "default_highway_time_factor": 1.0,
        "intersection_delay_s_by_street_count": {},
        "turn_delay_s": {"recto": 0.0, "giro": 0.0, "retorno": 0.0},
    },
    "2. Travel_time + tipo de via": {
        "intersection_delay_s_by_street_count": {},
        "turn_delay_s": {"recto": 0.0, "giro": 0.0, "retorno": 0.0},
    },
    "3. Travel_time + tipo de via + intersecciones": {
        "turn_delay_s": {"recto": 0.0, "giro": 0.0, "retorno": 0.0},
    },
    "4. Modelo completo": {},
}


Los perfiles se conservan como configuraciones del modelo; el selector utiliza el modelo completo.

Las metricas se recalculan y se muestran para cada seleccion de A y B.

La ruta final depende de la seleccion actual del menu.

El resumen de tiempo, distancia, ajustes y nodos explorados aparece junto a cada resultado.

## 6. Tabla geográfica de la ruta final

A diferencia de conectar solo los nodos, esta tabla conserva la geometría exacta de cada arista seleccionada por A*.


In [ ]:
def geometry_of_edge(u, v, key):
    edge_data = G.edges[u, v, key]
    geometry = edge_data.get("geometry")
    if geometry is not None:
        return geometry
    return LineString([
        (G.nodes[u]["x"], G.nodes[u]["y"]),
        (G.nodes[v]["x"], G.nodes[v]["y"]),
    ])


def build_route_geodata(result):
    """Crea la tabla geografica usando las aristas exactas de A*."""
    rows = []
    for order, ((u, v, key), detail) in enumerate(
        zip(result["route_edges"], result["edge_details"]), start=1
    ):
        rows.append({
            "orden": order,
            "node_start": u,
            "node_end": v,
            "edge_key": key,
            "length_m": detail["length_m"],
            "travel_time_s": detail["base_time_s"],
            "highway": detail["highway"],
            "speed_kph": detail["speed_kph"],
            "movement": detail["movement"],
            "total_edge_cost_s": detail["total_edge_cost_s"],
            "geometry": geometry_of_edge(u, v, key),
        })
    return gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")


## 7. Visualizaciones de la ruta

Primero se presenta la vista estática; luego el mapa interactivo y la animación.


In [ ]:
def plot_static_route(result, start_place, end_place):
    fig, ax = ox.plot_graph(
        G, figsize=(16, 16), node_size=0, edge_color="#C8C8C8",
        edge_linewidth=0.5, show=False, close=False,
    )
    for u, v, key in result["route_edges"]:
        geometry = G.edges[u, v, key].get("geometry")
        if geometry is None:
            x = [G.nodes[u]["x"], G.nodes[v]["x"]]
            y = [G.nodes[u]["y"], G.nodes[v]["y"]]
        else:
            x, y = geometry.xy
        ax.plot(x, y, color="#1565C0", linewidth=3, zorder=3)

    ax.scatter(start_place["lon"], start_place["lat"], color="#D32F2F",
               s=120, label=start_place["nombre"], zorder=4)
    ax.scatter(end_place["lon"], end_place["lat"], color="#2E7D32",
               s=120, label=end_place["nombre"], zorder=4)
    ax.set_title(f"Ruta A* completa: {result['total_cost_s'] / 60:.2f} min")
    ax.legend(fontsize=10)
    plt.show()


### Mapa interactivo: Mapbox o MapLibre

El notebook lee el token desde el archivo local `.env`. Si el archivo no existe o no contiene un token, el notebook utiliza MapLibre sin token. La rama Mapbox conserva `Scattermapbox`.


In [ ]:
def load_mapbox_token(env_path='.env'):
    """Lee MAPBOX_TOKEN desde .env sin imprimir ni exponer su valor."""
    path = Path(env_path)
    if not path.is_file():
        return None

    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        if key.strip() == 'MAPBOX_TOKEN':
            return value.strip().strip('"').strip("'") or None
    return None


MAPBOX_TOKEN = load_mapbox_token()
if MAPBOX_TOKEN and not MAPBOX_TOKEN.startswith('pk.'):
    raise ValueError('MAPBOX_TOKEN debe ser un token p?blico de Mapbox que comience por pk.')
print('Modo de mapa:', 'Mapbox desde .env' if MAPBOX_TOKEN else 'MapLibre sin token')


def route_coordinates(route_gdf):
    """Devuelve coordenadas lon/lat separadas por None para cada tramo."""
    lon, lat = [], []
    for geometry in route_gdf.geometry:
        coordinates = list(geometry.coords)
        lon.extend(point[0] for point in coordinates)
        lat.extend(point[1] for point in coordinates)
        lon.append(None)
        lat.append(None)
    return lon, lat


def route_center(route_gdf):
    min_x, min_y, max_x, max_y = route_gdf.total_bounds
    return {"lon": (min_x + max_x) / 2, "lat": (min_y + max_y) / 2}


def make_route_map(route_gdf, start_place, end_place, token=None):
    """Crea un mapa Mapbox si hay token; de lo contrario usa MapLibre."""
    lon, lat = route_coordinates(route_gdf)
    center = route_center(route_gdf)
    start_lon, start_lat = start_place["lon"], start_place["lat"]
    end_lon, end_lat = end_place["lon"], end_place["lat"]

    if token:
        figure = go.Figure([
            go.Scattermapbox(lon=lon, lat=lat, mode="lines", name="Ruta A*",
                             line={"width": 5, "color": "#1565C0"}),
            go.Scattermapbox(lon=[start_lon], lat=[start_lat], mode="markers",
                             name=start_place["nombre"], marker={"size": 14, "color": "#D32F2F"}),
            go.Scattermapbox(lon=[end_lon], lat=[end_lat], mode="markers",
                             name=end_place["nombre"], marker={"size": 14, "color": "#2E7D32"}),
        ])
        figure.update_layout(mapbox={
            "accesstoken": token,
            "style": "streets",
            "center": center,
            "zoom": 13,
        })
        provider = "Mapbox (streets)"
    else:
        figure = go.Figure([
            go.Scattermap(lon=lon, lat=lat, mode="lines", name="Ruta A*",
                          line={"width": 5, "color": "#1565C0"}),
            go.Scattermap(lon=[start_lon], lat=[start_lat], mode="markers",
                          name=start_place["nombre"], marker={"size": 14, "color": "#D32F2F"}),
            go.Scattermap(lon=[end_lon], lat=[end_lat], mode="markers",
                          name=end_place["nombre"], marker={"size": 14, "color": "#2E7D32"}),
        ])
        figure.update_layout(map={"style": "carto-positron", "center": center, "zoom": 13})
        provider = "MapLibre (carto-positron, sin token)"

    figure.update_layout(
        title=f"Ruta final A* — {provider}", height=650,
        margin={"r": 10, "t": 50, "l": 10, "b": 10},
        legend={"orientation": "h", "y": 0.02, "x": 0.01},
    )
    return figure


### Animación por tramos

La línea azul avanza sobre la geometría real de las aristas de la ruta final.


In [ ]:
def make_route_animation(route_gdf, start_place, end_place, token=None):
    """Anima el avance acumulado por cada tramo de la ruta final."""
    full_lon, full_lat = route_coordinates(route_gdf)
    center = route_center(route_gdf)
    start_lon, start_lat = start_place["lon"], start_place["lat"]
    end_lon, end_lat = end_place["lon"], end_place["lat"]
    trace_class = go.Scattermapbox if token else go.Scattermap

    frames, current_lon, current_lat = [], [], []
    for row in route_gdf.itertuples():
        coordinates = list(row.geometry.coords)
        current_lon.extend(point[0] for point in coordinates)
        current_lat.extend(point[1] for point in coordinates)
        frames.append(go.Frame(
            name=str(row.orden),
            data=[trace_class(lon=current_lon.copy(), lat=current_lat.copy(), mode="lines",
                              line={"width": 5, "color": "#1565C0"})],
            traces=[0],
        ))

    figure = go.Figure([
        trace_class(lon=full_lon, lat=full_lat, mode="lines", name="Recorrido",
                    line={"width": 5, "color": "rgba(21,101,192,0.20)"}),
        trace_class(lon=[start_lon], lat=[start_lat], mode="markers", name=start_place["nombre"],
                    marker={"size": 14, "color": "#D32F2F"}),
        trace_class(lon=[end_lon], lat=[end_lat], mode="markers", name=end_place["nombre"],
                    marker={"size": 14, "color": "#2E7D32"}),
    ], frames=frames)
    if token:
        figure.update_layout(mapbox={"accesstoken": token,
                                     "style": "streets",
                                     "center": center, "zoom": 13})
    else:
        figure.update_layout(map={"style": "carto-positron", "center": center, "zoom": 13})

    figure.update_layout(
        title="Animación de la ruta final por tramos", height=650,
        margin={"r": 10, "t": 50, "l": 10, "b": 10},
        updatemenus=[{"type": "buttons", "showactive": False, "x": 0.05, "y": 0.02,
                      "buttons": [{"label": "Reproducir", "method": "animate",
                                   "args": [None, {"frame": {"duration": 180, "redraw": True},
                                                   "fromcurrent": True}]}]}],
        sliders=[{"active": 0, "x": 0.16, "y": 0.02, "len": 0.78,
                  "steps": [{"label": frame.name, "method": "animate",
                             "args": [[frame.name], {"mode": "immediate",
                                                     "frame": {"duration": 0, "redraw": True}}]}
                            for frame in frames]}],
    )
    return figure


## 8. Selector de rutas estandarizadas

Seleccione dos lugares distintos. El boton calcula nuevamente la ruta A* con el modelo completo y actualiza el tiempo, la tabla geografica, los mapas y los enlaces de comparacion.
 
**Nota:** hagan click en Borrar todas las salidas de Jupyter para no guardar el token de Mapbox en el notebook.


In [ ]:
def selected_place(key):
    return dict(LUGARES_VALIDOS[key])


def nearest_valid_node(place):
    min_lon, min_lat, max_lon, max_lat = MAP_BOUNDS
    if not (min_lat <= place["lat"] <= max_lat and min_lon <= place["lon"] <= max_lon):
        raise ValueError(f"{place['nombre']} esta fuera de los limites del mapa.")
    node, distance_m = ox.distance.nearest_nodes(
        G, place["lon"], place["lat"], return_dist=True
    )
    if distance_m > MAX_SNAP_DISTANCE_M:
        raise ValueError(
            f"{place['nombre']} no tiene una via transitable a menos de "
            f"{MAX_SNAP_DISTANCE_M} m."
        )
    return node, float(distance_m)


def external_comparison_links(start_place, end_place):
    google_params = urlencode({
        "api": 1,
        "origin": f"{start_place['lat']},{start_place['lon']}",
        "destination": f"{end_place['lat']},{end_place['lon']}",
        "travelmode": "driving",
    })
    google_url = f"https://www.google.com/maps/dir/?{google_params}"
    waze_url = (
        "https://www.waze.com/ul?ll="
        f"{end_place['lat']}%2C{end_place['lon']}&navigate=yes"
    )
    return HTML(
        "<p><b>Puntos estandarizados:</b><br>"
        f"A - {start_place['nombre']}: {start_place['lat']}, {start_place['lon']}<br>"
        f"B - {end_place['nombre']}: {end_place['lat']}, {end_place['lon']}</p>"
        f'<p><a href="{google_url}" target="_blank">Abrir ruta en Google Maps</a><br>'
        f'<a href="{waze_url}" target="_blank">Abrir destino en Waze</a></p>'
    )


def calculate_selected_route(_=None):
    start_key = origin_dropdown.value
    end_key = destination_dropdown.value
    profile_name = profile_dropdown.value
    with route_output:
        clear_output(wait=True)
        if start_key == end_key:
            display(HTML("<b>Seleccione dos lugares distintos para A y B.</b>"))
            return
        try:
            start_place = selected_place(start_key)
            end_place = selected_place(end_key)
            start_id, start_snap_m = nearest_valid_node(start_place)
            end_id, end_snap_m = nearest_valid_node(end_place)

            # configurar Node, crear raiz y buscar con Tree.
            OSMRouteNode.configure(G, perfiles_costo[profile_name])
            root = OSMRouteNode(
                state=(None, start_id, None),
                value="0",
                objective=end_id,
                accumulated_cost=0.0,
            )
            tree = OSMRouteTree(root)
            goal_node = tree.aStar(end_id)
            result = result_from_goal(goal_node, tree)

            validate_composite_route(result, start_id, end_id)
            route_gdf = build_route_geodata(result)
        except (ValueError, AssertionError) as error:
            display(HTML(f"<b>No se pudo calcular la ruta:</b> {error}"))
            return

        display(external_comparison_links(start_place, end_place))
        display(HTML(
            f"<b>Perfil:</b> {profile_name}<br>"
            f"<b>Estructura:</b> OSMRouteNode(Node) + OSMRouteTree(Tree)"
        ))
        display(pd.DataFrame({"Metricas de la mejor ruta": summarize_composite_result(result)}))
        display(pd.DataFrame({
            "nodo vial": [start_id, end_id],
            "distancia al nodo (m)": [round(start_snap_m, 1), round(end_snap_m, 1)],
        }, index=["A", "B"]))
        display(route_gdf.drop(columns="geometry").head())
        plot_static_route(result, start_place, end_place)
        display(make_route_map(route_gdf, start_place, end_place, MAPBOX_TOKEN))
        display(make_route_animation(route_gdf, start_place, end_place, MAPBOX_TOKEN))


place_options = [(place["nombre"], key) for key, place in LUGARES_VALIDOS.items()]
origin_dropdown = widgets.Dropdown(options=place_options, description="Punto A:")
destination_dropdown = widgets.Dropdown(options=place_options, description="Punto B:")
profile_dropdown = widgets.Dropdown(
    options=list(perfiles_costo),
    value="4. Modelo completo",
    description="Costo:",
)
calculate_button = widgets.Button(
    description="Calcular ruta", button_style="primary", icon="route"
)
route_output = widgets.Output()
calculate_button.on_click(calculate_selected_route)

display(widgets.VBox([
    widgets.HTML(
        "<h3>Seleccion de puntos estandarizados</h3>"
        "La ruta se calcula con la estructura Node y Tree del curso."
    ),
    origin_dropdown,
    destination_dropdown,
    profile_dropdown,
    calculate_button,
    route_output,
]))


## Conclusion

La ruta final se obtiene con la estructura que vimos en clase para búsquedas a ciegas: un nodo especializado que hereda de `Node` y un arbol especializado que hereda de `Tree`. A* sigue minimizando un costo temporal compuesto y mantiene la heuristica geografica admisible.